# Phase 8 - QLoRA Fine-Tuning (Colab)

Fine-tunes **Qwen3-4B-Instruct-2507** on the Phase-7 RAFT dataset so it emits safe,
grounded, correctly-structured guidance **directly from the COMPACT prompt** -
internalizing the format + safety rules the validator currently patches.

```text
4-bit NF4 (double-quant) + LoRA r=16 / alpha=32 / dropout 0.05
targets: q,k,v,o,gate,up,down_proj
loss masked to the assistant turn only
3 epochs, lr 2e-4 cosine, best checkpoint on val loss
```

**Runtime:** `Runtime -> Change runtime type -> GPU`. A T4 is enough in 4-bit;
A100 is faster. The script auto-detects VRAM and picks batch / seq-len defaults.

**Prerequisite:** `train.jsonl` + `val.jsonl` from Phase 7
(`python -m llm.data.generate_dataset ...`).

Order: install -> repo -> data -> **dry-run** -> train -> curves -> eval -> download.


In [ ]:
# ============================================================
# Cell 1 - Dependencies  (torch is preinstalled on Colab)
# ============================================================
%pip install -q "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" \
               "accelerate>=0.33" datasets matplotlib

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  ({p.total_memory/1024**3:.1f} GB)')


## 2. Get the project code

Either clone the repo (set `REPO_URL`) **or** upload/copy the project folder to
`/content/Nephro-AI`. The scripts are run as modules from the repo root.


In [ ]:
# ============================================================
# Cell 2 - Project code on the path
# ============================================================
import os, sys
from pathlib import Path

REPO_URL = ''            # <-- set to your git remote, or leave '' if uploading
REPO = Path('/content/Nephro-AI')

if not REPO.exists():
    if REPO_URL:
        !git clone -q {REPO_URL} {REPO}
    else:
        raise SystemExit('Set REPO_URL, or upload the project to /content/Nephro-AI')

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('cwd:', Path.cwd())

# sanity: the Phase-8 package imports (pure-python, no torch needed)
from llm.finetune.data import load_examples, suggest_max_seq_len
print('llm.finetune imported OK')


## 3. Phase-7 dataset

Upload `train.jsonl` / `val.jsonl` into `phase7_data/`, or mount Drive if they
already live there.


In [ ]:
# ============================================================
# Cell 3 - Load the Phase-7 data
# ============================================================
from pathlib import Path
DATA = Path('phase7_data'); DATA.mkdir(exist_ok=True)

# --- Option A: upload from your machine ---
if not (DATA/'train.jsonl').exists():
    from google.colab import files
    print('Select train.jsonl and val.jsonl ...')
    for name, blob in files.upload().items():
        (DATA/name).write_bytes(blob)

# --- Option B: Google Drive (uncomment) ---
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/phase7_data/*.jsonl {DATA}/

from llm.finetune.data import load_examples
tr = load_examples(DATA/'train.jsonl')
va = load_examples(DATA/'val.jsonl') if (DATA/'val.jsonl').exists() else []
print(f'train={len(tr)}  val={len(va)}')

from collections import Counter
print('class mix :', dict(Counter(r['risk_class'] for r in tr)))
print('teachers  :', dict(Counter(r.get('teacher','?') for r in tr)))


## 4. Dry-run first (no GPU work)

Encodes the data, applies assistant-only masking and prints token-length stats.

**Check `truncated` is 0.** If not, raise `--max-seq-len` (or pass `--auto-seq-len`).


In [ ]:
# ============================================================
# Cell 4 - Dry-run: validate the data pipeline
# ============================================================
!python -m llm.finetune.qlora_train --data-dir phase7_data --dry-run


## 5. Train

Batch / accumulation / seq-len default from detected VRAM; override with
`--batch`, `--accum`, `--max-seq-len` if you hit OOM.

On a T4 with ~500 examples expect roughly 30-60 min for 3 epochs.


In [ ]:
# ============================================================
# Cell 5 - QLoRA fine-tune
# ============================================================
OUT = 'llm/finetune/adapters/v1'

!python -m llm.finetune.qlora_train \
    --data-dir phase7_data \
    --out-dir {OUT} \
    --epochs 3 --lr 2e-4 --lora-r 16 --lora-alpha 32

# OOM? re-run with:  --batch 1 --accum 16 --max-seq-len 1536


In [ ]:
# ============================================================
# Cell 6 - Training curves
# ============================================================
import pandas as pd
from IPython.display import Image, display

log = pd.read_csv(f'{OUT}/training_log.csv')
display(log.tail(12))

png = Path(OUT)/'training_curve.png'
if png.exists():
    display(Image(str(png)))

print(open(f'{OUT}/run_card.md', encoding='utf-8').read())


## 7. Acceptance check - the headline metric

Generates on `val.jsonl` from the **compact** prompt with the **validator OFF**.

`raw_unsafe_rate` is the number that matters: Phase-6 base scored **0.10** (with the
validator patching it). Target here is **~0 without any validator**.

Run the base arm too for the Phase-9 ablation pair.


In [ ]:
# ============================================================
# Cell 7 - Evaluate: fine-tuned arm (and base arm for comparison)
# ============================================================
!python -m llm.finetune.evaluate_adapter \
    --data-dir phase7_data --adapter {OUT}/adapter --tag finetuned

# Base arm (no adapter) - the comparison point:
# !python -m llm.finetune.evaluate_adapter --data-dir phase7_data --tag base


In [ ]:
# ============================================================
# Cell 8 - Download the adapter + artifacts
# ============================================================
!zip -qr adapter_v1.zip {OUT} phase8_eval
print('zipped')

from google.colab import files
files.download('adapter_v1.zip')


## Notes

**Serving** - no runtime changes needed; the adapter loads through the existing hook:

```python
from llm.serving.model_backends import QwenChatModel
model = QwenChatModel(adapter_path='llm/finetune/adapters/v1/adapter')
```

**Troubleshooting**

| Symptom | Fix |
|---|---|
| CUDA OOM | `--batch 1 --accum 16`, then `--max-seq-len 1536` |
| `truncated > 0` in dry-run | raise `--max-seq-len`, or `--auto-seq-len` |
| val loss rises after epoch 1-2 | overfitting - `--epochs 2`; best checkpoint is kept anyway |
| bitsandbytes import error | restart runtime after the pip install |

**Next:** Phase 9 ablation - fine-tuned vs the frozen Phase-6 base arm
(raw_unsafe down, prompt-length / inference cost down).
